# CIFAR-10 with JEPA - we will do a comparison with old approach.
JEPA uses the concept of embeddings and training on them to eventually "compress" picture information into a smaller dimension.
This reduces memory usage and compute time. With GPUs being very expensive now, it is good to get on a smaller one if you can.
JEPA's advantages only start happening with larger image sizes and datasets. CIFAR-10 is more complex but still small enough 
to reasonably look at. It has 10 different image classes like we had with MNIST but the pictures are color (3 channels) and slightly higher resolution (32x32).
# CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

In [ ]:
# Standard imports and random seeds
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
# This is complex code not shown but you can get from github.
from jepa_utils import (JEPA, extract_features, probe_accuracy,
                        dense_baseline, print_class_balance)

SEED = 42
FRACTIONS = [1.0, 0.2, 0.05, 0.01]   # percentage of training data that will be used.
EPOCHS_JEPA, EPOCHS_PROBE, EPOCHS_SUP = 25, 30, 30
tf.random.set_seed(SEED); np.random.seed(SEED)

# Let's talk about FRACTIONS.
# This is a small percentage of total images. This is to demonstrate JEPA's usefulness case.
# More labeled examples generally means dense would do better.
# Also worth noting that 3 channel pictures were analyzed with convolutional neural networks (CNN) which are better than Dense for 3 color picture analysis. THIS IS A DEMONSTRATION OF A POSSIBLE USE CASE if you had a lot of data but only a small portion is labeled.

In [ ]:
# We start out with familiar pipeline, like before dataset will automatically download.
cifar10 = keras.datasets.cifar10
# divide into train and test
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()
# Do scaling like before to get our data between 0 and 1
min_val, max_val = x_train.min(), x_train.max()
train_images = ((x_train - min_val) / (max_val - min_val)).astype("float32")
test_images = ((x_test - min_val) / (max_val - min_val)).astype("float32")

In [ ]:
# We should always check if dataset is balanced
print_class_balance(y_train, y_test)

# JEPA variables we want to set:
JEPA uses grids to stride over the entire image and turn it into embeddings.
IMG and CHANNELS are the dimensions of CIFAR pictures.

In [ ]:
x_train.shape

In [ ]:
# x_train.shape is (50000, 32, 32, 3) - 50k count of 32 x 32 x 3 images
# 4 = side of our 4x4 grid.
# 128 is a somewhat arbitrary number picked for embedding dimension.
IMG, CHANNELS, PATCH, EMB = x_train.shape[2], x_train.shape[3], 4, 128
GRID, PATCH_DIM = IMG // PATCH, PATCH * PATCH * CHANNELS

In [ ]:
# JEPA: self-supervised (NO LABELS) pretraining
jepa = JEPA(img=IMG, channels=CHANNELS, patch=PATCH, emb=EMB)
jepa.fit(train_images, epochs=EPOCHS_JEPA, batch=256, seed=SEED)  # NOTE NO LABELS in this line
# Encoder is trained on images, then frozen afterward to enable proper testing as we have mentioned previously.

# Trains a single small Dense layer network on 128 features -> 10 (is size of layer) from the encoder for each image.
train_feats = extract_features(jepa.context_encoder, train_images, GRID, PATCH, CHANNELS)
test_feats  = extract_features(jepa.context_encoder, test_images,  GRID, PATCH, CHANNELS)
# NOTE: loss printed is mse, not a classification type loss

In [ ]:
#Loop to compare JEPA and simple DENSE with different fractions.
acc_dense, acc_jepa = [], []
print("\n fraction |  dense (raw px) |  jepa + probe")
print(" ---------+-----------------+---------------")
for frac in FRACTIONS:
    d = dense_baseline(train_images, y_train, test_images, y_test,
                       IMG, CHANNELS, label_fraction=frac,
                       epochs=EPOCHS_SUP, seed=SEED)
    p = probe_accuracy(train_feats, y_train, test_feats, y_test, EMB,
                       label_fraction=frac, epochs=EPOCHS_PROBE, seed=SEED)
    acc_dense.append(d * 100)
    acc_jepa.append(p * 100)
    print(f"   {frac:5.2f}  |     {d * 100:5.2f}%     |    {p * 100:5.2f}%")

In [ ]:
# Plot accuracies across all FRACTIONS and both technique
plt.figure(figsize=(6, 4))
plt.plot(FRACTIONS, acc_dense, color='blue', marker='o',
         label='Dense net (raw pixels)')
plt.plot(FRACTIONS, acc_jepa, color='green', marker='o',
         label='JEPA pretraining + linear probe')
plt.xscale('log')
plt.xlabel('Fraction of labeled training images')
plt.ylabel('Test accuracy (%)')
plt.title('Accuracy vs. labeled fraction (CIFAR-10)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_vs_label_fraction.png', dpi=150)
plt.show()